# Semana 6: Logística, métricas y umbral

**Pregunta de trabajo.** Comprobar sigmoide, gradiente y elección de umbral en dev antes de una única medición en test.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
data = load_breast_cancer()
X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    data.data, data.target, test_size=0.20, random_state=2105, stratify=data.target
)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev, test_size=0.25, random_state=2105, stratify=y_train_dev
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
model.fit(X_train, y_train)
p_dev = model.predict_proba(X_dev)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]
print({"train": len(y_train), "dev": len(y_dev), "test": len(y_test)})

{'train': 341, 'dev': 114, 'test': 114}


In [3]:
for threshold in [0.3, 0.5, 0.7]:
    pred = (p_dev >= threshold).astype(int)
    print(threshold, accuracy_score(y_dev, pred), precision_score(y_dev, pred), recall_score(y_dev, pred), f1_score(y_dev, pred))
print(confusion_matrix(y_dev, p_dev >= 0.5))

0.3 0.9912280701754386 0.9861111111111112 1.0 0.993006993006993
0.5 0.9912280701754386 0.9861111111111112 1.0 0.993006993006993
0.7 0.9912280701754386 1.0 0.9859154929577465 0.9929078014184397
[[42  1]
 [ 0 71]]


## Gradiente, baseline y selección del umbral

La partición dev selecciona el umbral. Test se consulta una sola vez después de congelar modelo y regla.

In [4]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
{"baseline_f1_dev": f1_score(y_dev, baseline.predict(X_dev)), "n_dev": len(y_dev), "n_test": len(y_test)}

{'baseline_f1_dev': 0.7675675675675676, 'n_dev': 114, 'n_test': 114}

In [5]:
def stable_sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    positive = z >= 0
    out[positive] = 1 / (1 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1 + exp_z)
    return out

X_small = np.array([[1.0, -1.0], [1.0, 0.5], [1.0, 2.0]])
y_small = np.array([0.0, 1.0, 1.0])
theta_small = np.array([0.2, -0.3])
analytic = X_small.T @ (stable_sigmoid(X_small @ theta_small) - y_small) / len(y_small)
eps = 1e-6
def logistic_loss(theta):
    p = np.clip(stable_sigmoid(X_small @ theta), 1e-12, 1 - 1e-12)
    return -np.mean(y_small * np.log(p) + (1 - y_small) * np.log(1 - p))
numeric = np.array([(logistic_loss(theta_small + eps * np.eye(2)[j]) - logistic_loss(theta_small - eps * np.eye(2)[j])) / (2 * eps) for j in range(2)])
{"analytic": analytic, "numeric": numeric, "error": np.linalg.norm(analytic - numeric)}

{'analytic': array([-0.15457698, -0.68786198]),
 'numeric': array([-0.15457698, -0.68786198]),
 'error': 4.1997711856989656e-11}

In [6]:
threshold_rows = []
for threshold in np.arange(0.2, 0.81, 0.1):
    pred = (p_dev >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_dev, pred).ravel()
    threshold_rows.append({
        "threshold": threshold, "precision": precision_score(y_dev, pred),
        "recall": recall_score(y_dev, pred), "f1": f1_score(y_dev, pred),
        "fp": fp, "fn": fn,
    })
threshold_results = pd.DataFrame(threshold_rows)
threshold_results

,threshold,precision,recall,f1,fp,fn
0,0.2,0.986111,1.000000,0.993007,1,0
1,0.3,0.986111,1.000000,0.993007,1,0
2,0.4,0.986111,1.000000,0.993007,1,0
3,0.5,0.986111,1.000000,0.993007,1,0
4,0.6,1.000000,1.000000,1.000000,0,0
5,0.7,1.000000,0.985915,0.992908,0,1
6,0.8,1.000000,0.985915,0.992908,0,1


In [7]:
chosen_threshold = float(threshold_results.loc[threshold_results["f1"].idxmax(), "threshold"])
pred_test = (p_test >= chosen_threshold).astype(int)
{
    "threshold_from_dev": chosen_threshold,
    "f1_test": f1_score(y_test, pred_test),
    "confusion_test": confusion_matrix(y_test, pred_test).tolist(),
}

{'threshold_from_dev': 0.6000000000000001,
 'f1_test': 0.9655172413793104,
 'confusion_test': [[39, 3], [2, 70]]}

In [8]:
assert np.linalg.norm(analytic - numeric) < 1e-6
assert confusion_matrix(y_test, pred_test).sum() == len(y_test)
print("Umbral fijado en dev; la cifra de test estima el procedimiento ya congelado.")

Umbral fijado en dev; la cifra de test estima el procedimiento ya congelado.
